In [11]:
import os
import requests
import time
import json

from pathlib import Path

### Import CBC Raw Predictions

In [8]:
CBC_PREDICTIONS_PATH = Path("../data/results/final/contemporary/contemporary_predictions.json")

with open(CBC_PREDICTIONS_PATH, "r") as f:
    cbc_predictions = json.load(f)

raw_entities = cbc_predictions["entities"]

In [ ]:
UMLS_API_KEY = os.getenv("UMLS_API_KEY")
UMLS_SEARCH_URL = "https://uts-ws.nlm.nih.gov/rest/search/current"

def lookup_umls(entity_text, api_key=UMLS_API_KEY, max_retries=5):
    params = {
        "string": entity_text,
        "apiKey": api_key,
        "searchType": "exact"  # falls back to normal search if you widen this later
    }

    for attempt in range(max_retries):
        try:
            resp = requests.get(UMLS_SEARCH_URL, params=params, timeout=10)
            if resp.status_code == 429:
                wait = 2 ** attempt
                print(f"Rate limited on '{entity_text}', waiting {wait}s...")
                time.sleep(wait)
                continue
            resp.raise_for_status()
            results = resp.json()["result"]["results"]
            if results and results[0]["ui"] != "NONE":
                return {"cui": results[0]["ui"], "name": results[0]["name"]}
            return None
        except Exception as e:
            print(f"Lookup failed for '{entity_text}': {e}")
            return None
    return None


# ---------- Apply to your raw_entities dict ----------
for entity_id, entity in raw_entities.items():
    result = lookup_umls(entity["text"])
    entity["cui"] = result["cui"] if result else None
    entity["umls_name"] = result["name"] if result else None
    time.sleep(0.1)  # be polite to the API